# Autoencoder Representation Learning

A compact PyTorch autoencoder experiment for nonlinear representation learning and reconstruction from synthetically generated high-dimensional data.

> Portfolio-ready copy of the original completed experiment. Experimental code and saved outputs are preserved; assignment-administration text has been removed.


In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset, random_split

In [ ]:
# -----------------------------
# 1) Reproducibility / Device
# -----------------------------
torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda


In [ ]:
# -----------------------------
# 2) Dataset (AS GIVEN)
# -----------------------------
n_samples = 20000
input_dim = 100
latent_dim = 16

true_latent = torch.randn(n_samples, latent_dim)
mixing = torch.randn(latent_dim, input_dim)
X = true_latent @ mixing + 0.1 * torch.randn(n_samples, input_dim)

In [ ]:
# Normalize feature-wise
X = (X - X.mean(dim=0)) / (X.std(dim=0) + 1e-6)
print("X shape:", X.shape)  # (20000, 100)

X shape: torch.Size([20000, 100])


In [ ]:
# Wrap in TensorDataset (targets are same as inputs for AE)
dataset = TensorDataset(X)

In [ ]:
# -----------------------------
# 3) Train/Val/Test splits
# -----------------------------
train_ratio, val_ratio, test_ratio = 0.80, 0.10, 0.10
n_total = len(dataset)
n_train = int(train_ratio * n_total)
n_val = int(val_ratio * n_total)
n_test = n_total - n_train - n_val

train_ds, val_ds, test_ds = random_split(
    dataset,
    [n_train, n_val, n_test],
    generator=torch.Generator().manual_seed(42)
)

batch_size = 256
train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, drop_last=False)
val_loader   = DataLoader(val_ds, batch_size=batch_size, shuffle=False, drop_last=False)
test_loader  = DataLoader(test_ds, batch_size=batch_size, shuffle=False, drop_last=False)

print(f"Split sizes: train={len(train_ds)}, val={len(val_ds)}, test={len(test_ds)}")

Split sizes: train=16000, val=2000, test=2000


In [ ]:
# -----------------------------
# 4) Autoencoder Architecture
# -----------------------------
class Autoencoder(nn.Module):
    def __init__(self, input_dim=100, latent_dim=16):
        super().__init__()
        # Encoder: input -> latent
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, latent_dim)
        )
        # Decoder: latent -> input
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 32),
            nn.ReLU(),
            nn.Linear(32, 64),
            nn.ReLU(),
            nn.Linear(64, input_dim)
        )

    def forward(self, x):
        z = self.encoder(x)
        x_hat = self.decoder(z)
        return x_hat

model = Autoencoder(input_dim=input_dim, latent_dim=latent_dim).to(device)

In [ ]:
# -----------------------------
# 5) Training parameters
# -----------------------------
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)

epochs = 30


In [ ]:
# -----------------------------
# 6) Train AE (report loss)
# -----------------------------
def run_epoch(loader, training: bool):
    if training:
        model.train()
    else:
        model.eval()

    total_loss = 0.0
    total_count = 0

    for (x_batch,) in loader:
        x_batch = x_batch.to(device)

        if training:
            optimizer.zero_grad()

        with torch.set_grad_enabled(training):
            x_hat = model(x_batch)
            loss = criterion(x_hat, x_batch)

            if training:
                loss.backward()
                optimizer.step()

        bs = x_batch.size(0)
        total_loss += loss.item() * bs
        total_count += bs

    return total_loss / total_count

best_val = float("inf")
best_state = None

for epoch in range(1, epochs + 1):
    train_loss = run_epoch(train_loader, training=True)
    val_loss = run_epoch(val_loader, training=False)

    if val_loss < best_val:
        best_val = val_loss
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    if epoch == 1 or epoch % 5 == 0 or epoch == epochs:
        print(f"Epoch {epoch:02d}/{epochs} | Train MSE: {train_loss:.6f} | Val MSE: {val_loss:.6f}")

# Load best model (based on validation loss)
if best_state is not None:
    model.load_state_dict(best_state)
    model.to(device)


Epoch 01/30 | Train MSE: 0.878261 | Val MSE: 0.634748
Epoch 05/30 | Train MSE: 0.267432 | Val MSE: 0.245739
Epoch 10/30 | Train MSE: 0.144457 | Val MSE: 0.144653
Epoch 15/30 | Train MSE: 0.093731 | Val MSE: 0.086374
Epoch 20/30 | Train MSE: 0.036545 | Val MSE: 0.032249
Epoch 25/30 | Train MSE: 0.024854 | Val MSE: 0.025325
Epoch 30/30 | Train MSE: 0.022155 | Val MSE: 0.020889


In [ ]:
# -----------------------------
# 7) Evaluate + Reconstruct on test set
# -----------------------------
model.eval()
test_mse = run_epoch(test_loader, training=False)
print(f"\nHeld-out TEST MSE (reconstruction): {test_mse:.6f}")

# Collect a small batch of reconstructions for inspection
with torch.no_grad():
    (x_batch,) = next(iter(test_loader))
    x_batch = x_batch.to(device)
    x_hat = model(x_batch)

# Bring to CPU for printing / further analysis
x_batch_cpu = x_batch[:5].cpu()
x_hat_cpu = x_hat[:5].cpu()

print("\nExample reconstructions (first 5 samples, first 10 features):")
for i in range(5):
    original = x_batch_cpu[i, :10]
    recon    = x_hat_cpu[i, :10]
    per_sample_mse = torch.mean((recon - original) ** 2).item()
    print(f"\nSample {i} | per-sample MSE: {per_sample_mse:.6f}")
    print("Original:", original.numpy())
    print("Recon   :", recon.numpy())


Held-out TEST MSE (reconstruction): 0.020188

Example reconstructions (first 5 samples, first 10 features):

Sample 0 | per-sample MSE: 0.043068
Original: [-1.8630944   0.7336565   1.0526966  -0.9782695  -1.0613593  -1.4211698
 -0.3726711  -1.0490211  -0.67892474 -1.588363  ]
Recon   : [-1.8362901   0.40802577  0.9787811  -0.92741156 -0.8423213  -1.1038282
 -0.19144864 -1.3547196  -0.50508934 -1.4850081 ]

Sample 1 | per-sample MSE: 0.004342
Original: [ 0.36209726  0.10448959 -0.11426738  0.6904327  -0.01643649  0.03302435
 -0.4811733   0.8505512  -0.66203105 -0.40601486]
Recon   : [ 0.3894664   0.01417097 -0.12612894  0.67489254 -0.02041636  0.12630159
 -0.46550316  0.716513   -0.5780588  -0.39367932]

Sample 2 | per-sample MSE: 0.016684
Original: [-0.09886748  1.7596402   0.42936686 -0.19152011 -1.002945   -2.3919132
 -0.85612786 -0.01454693 -0.73545295  1.4043928 ]
Recon   : [-0.18284117  1.5298648   0.4752601  -0.15795808 -0.8968351  -2.264728
 -0.7525892  -0.19048117 -0.5846949  